<a href="https://colab.research.google.com/github/Arx15E/IntegracionDatos/blob/main/Fallas_tecnologicas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# ================================
# RETO: Evaluación de Riesgo Operacional en Canales Electrónicos
# ================================

# 1. Instalar librerías necesarias
!pip install pandas numpy openpyxl plotly dash jupyter-dash -q

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente")

# ============================
# 2. CARGAR LOS DATOS
# ============================
# Asegúrate de que tu archivo Excel esté en la ruta '/content/5. Riesgo Operacional FallasTecnológicas.xlsx'
file_path = '/content/5. Riesgo Operacional FallasTecnológicas.xlsx'
print(f"Cargando datos desde: {file_path}")

# Cargar hojas
df_fallas = pd.read_excel(file_path, sheet_name='Fallas Tecnológicas', header=1)
df_riesgos = pd.read_excel(file_path, sheet_name='Riesgos')
df_gestion = pd.read_excel(file_path, sheet_name='Gestión')

# Rename columns after loading
df_riesgos = df_riesgos.rename(columns={'Unnamed: 1': 'Nivel de Impacto',
                                         'Unnamed: 2': 'Descripción del Fallo',
                                         'Unnamed: 3': 'Impacto Financiero (USD)'})
df_gestion = df_gestion.rename(columns={'Unnamed: 1': 'Nivel de Gestión'})

print(f"📊 Dimensiones Fallas: {df_fallas.shape}")
print(f"📊 Dimensiones Riesgos: {df_riesgos.shape}")

# ============================
# 3. LIMPIEZA Y PREPARACIÓN
# ============================
# Convertir fecha serial de Excel a datetime
df_fallas['Fechas'] = pd.to_datetime(df_fallas['Fechas'], errors='coerce')

# Mapear descripciones a niveles de riesgo
nivel_map = {
    'Errores visuales menores': 1,
    'Errores en la interfaz de usuario': 2,
    'Degradación del rendimiento por varias horas': 3,
    'Degradación severa del rendimiento por más de 12 horas': 4,
    'Caída total del sistema por más de 24 horas': 5
}

def get_nivel(descripcion):
    for key, nivel in nivel_map.items():
        if key in str(descripcion):
            return nivel
    return 1  # default

df_fallas['Nivel_Riesgo'] = df_fallas['Descripción Evento'].apply(get_nivel)

# Renombrar columnas para mayor claridad
df_fallas = df_fallas.rename(columns={
    'Transacciones Diarias': 'Transacciones',
    'Valor Transado (millones)': 'Valor_Transado_M',
    'Transacciones Fallidas': 'Fallidas',
    'Valor Generado Promedio (Millones)': 'Valor_Perdida_M'
})

print("✅ Datos preparados")

# ============================
# 4. MATRICES REQUERIDAS
# ============================

# Matriz de Frecuencia
frecuencia = df_fallas['Nivel_Riesgo'].value_counts().sort_index()
frecuencia.name = 'Frecuencia'

# Matriz de Severidad (usando niveles)
severidad = df_riesgos.set_index('Nivel de Impacto')[['Descripción del Fallo', 'Impacto Financiero (USD)']]

# Pérdidas Agregadas
perdidas_agregadas = pd.DataFrame({
    'Nivel': range(1,6),
    'Frecuencia': frecuencia.reindex(range(1,6), fill_value=0),
    'Pérdida_Promedio_M': df_fallas.groupby('Nivel_Riesgo')['Valor_Perdida_M'].mean().reindex(range(1,6), fill_value=0)
})
perdidas_agregadas['Pérdida_Total_M'] = perdidas_agregadas['Frecuencia'] * perdidas_agregadas['Pérdida_Promedio_M']

print("\n📈 Matriz de Pérdidas Agregadas:")
print(perdidas_agregadas)

# ============================
# 5. ESTIMACIÓN CUANTITATIVA
# ============================

# Pérdida Esperada (PE)
pe = perdidas_agregadas['Pérdida_Total_M'].sum()
print(f"\n💰 Pérdida Esperada (PE): ${pe:,.2f} millones")

# OpVaR 99.9% - Simulación Monte Carlo simple
np.random.seed(42)
n_sim = 10000

# Distribución de frecuencia (Poisson) y severidad (LogNormal aproximada)
frec_sim = np.random.poisson(
    lam=perdidas_agregadas['Frecuencia'].sum(),
    size=n_sim
)
severidad_sim = np.random.lognormal(
    mean=np.log(perdidas_agregadas['Pérdida_Promedio_M'].mean()),
    sigma=1.0,
    size=n_sim
)

perdidas_sim = frec_sim * severidad_sim

opvar_999 = np.percentile(perdidas_sim, 99.9)
print(f"⚠️ OpVaR 99.9%: ${opvar_999:,.2f} millones")

# Escenarios: Inherente vs Residual
# (Aproximación simple: gestión reduce pérdida en un porcentaje)
reduccion_gestion = 0.35  # 35% de mitigación
opvar_residual = opvar_999 * (1 - reduccion_gestion)

print(f"🛡️ OpVaR Residual (con gestión): ${opvar_residual:,.2f} millones")

# ============================
# 6. VISUALIZACIONES INTERACTIVAS
# ============================

# Gráfico 1: Distribución por Nivel de Riesgo
fig1 = px.bar(
    perdidas_agregadas,
    x='Nivel',
    y='Pérdida_Total_M',
    title='Pérdidas Agregadas por Nivel de Riesgo',
    labels={'Pérdida_Total_M': 'Pérdida Total (Millones USD)', 'Nivel': 'Nivel de Riesgo'},
    color='Pérdida_Total_M',
    color_continuous_scale='Reds'
)
fig1.show()

# Gráfico 2: Evolución Temporal
df_fallas['Mes'] = df_fallas['Fechas'].dt.to_period('M')
evolucion = df_fallas.groupby('Mes')['Valor_Perdida_M'].sum().reset_index()
evolucion['Mes'] = evolucion['Mes'].astype(str)

fig2 = px.line(evolucion, x='Mes', y='Valor_Perdida_M',
               title='Evolución de Pérdidas por Mes')
fig2.show()

# ============================
# 7. HERRAMIENTA DE AUDITORÍA (Filtro Dinámico)
# ============================

from ipywidgets import interact, Dropdown, Output
import ipywidgets as widgets

out = Output()

def mostrar_eventos(nivel):
    with out:
        out.clear_output()
        eventos = df_fallas[df_fallas['Nivel_Riesgo'] == nivel]
        print(f"\n🔍 Eventos de Nivel {nivel}:")
        display(eventos[['Fechas', 'Fallidas', 'Valor_Perdida_M', 'Descripción Evento']].head(10))

nivel_selector = Dropdown(
    options=[1,2,3,4,5],
    value=1,
    description='Nivel de Riesgo:'
)

print("🎛️ Herramienta de Auditoría Interactiva")
interact(mostrar_eventos, nivel=nivel_selector)
display(out)

✅ Librerías cargadas correctamente
Cargando datos desde: /content/5. Riesgo Operacional FallasTecnológicas.xlsx
📊 Dimensiones Fallas: (701, 6)
📊 Dimensiones Riesgos: (6, 7)
✅ Datos preparados

📈 Matriz de Pérdidas Agregadas:
              Nivel  Frecuencia  Pérdida_Promedio_M  Pérdida_Total_M
Nivel_Riesgo                                                        
1                 1         280            0.542377         151.8656
2                 2         251            1.547616         388.4516
3                 3         115            3.304257         379.9896
4                 4          40            6.661038         266.4415
5                 5          15           11.057980         165.8697

💰 Pérdida Esperada (PE): $1,352.62 millones
⚠️ OpVaR 99.9%: $64,356.09 millones
🛡️ OpVaR Residual (con gestión): $41,831.46 millones


🎛️ Herramienta de Auditoría Interactiva


interactive(children=(Dropdown(description='Nivel de Riesgo:', options=(1, 2, 3, 4, 5), value=1), Output()), _…

Output()